# Validation for a new model

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

from model_plots import (
    plot_et_comparison,
    plot_variable_evolution,
    print_error_metrics,
)
from model_tools import (
    et_sd_list,
    extra_variable_list,
    find_directories,
    radiation_list,
)


## Input Data

We obtained the observed evapotranspiration, daily radiation, and extra ERA5-Land variables using the **prepare_et_single_date**, **prepare_daily_radiation**, and **prepare_daily_explanatory** functions from **ET-DATASET**, respectively. The data were then stored for each region of interest in a dedicated subdirectory within **SWEAT_TEST_DATA_PATH**.


In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

In [ ]:
input_dir = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "senegal"
)
dir_sd, dir_rad, dir_var = find_directories(input_dir)

In [ ]:
min_date = "2024-03-01"
max_date = "2024-03-31"
x = 345700
y = 1590936
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)

In [ ]:
plot_variable_evolution(dir_var, dir_sd, dir_rad, min_date, max_date, x, y)

### Test on a dummy time series generation algorithm.

In [ ]:
# dummy algorithm
def et_ts_list(
    dir_rad: str,
    dir_sd: str,
    dir_var: str,
    start_date,
    end_date,
    x: int,
    y: int,
    freq: int,
) -> list[float]:
    """
    Computes the simulated daily evapotranspiration at a given pixel
    over a specified period using a dummy algorithm (mean of all variables)

    Parameters
    ----------
    dir_rad: str
        Directory where the `radiation_YYYYMMDD.tif` files are stored
    dir_sd: str
        Directory where the `et_single_date_YYYYMMDD.tif` are stored
    dir_var: str
        Directory where the `explanatory_YYYYMMDD.tif` are stored
    start_date: dt.datetime
        Start date
    end_date: dt.datetime
        End date
    x: int
        First axis coordinate (UTM)
    y: int
        Second axis coordinate (UTM)
    freq: int
        Acquisition frequency

    Returns
    -------
    list[float]
        Simulated daily evapotranspiration
    """
    tp = extra_variable_list(dir_var, "tp", start_date, end_date, x, y)
    sw = extra_variable_list(dir_var, "sw", start_date, end_date, x, y)
    sro = extra_variable_list(dir_var, "sro", start_date, end_date, x, y)
    src = extra_variable_list(dir_var, "src", start_date, end_date, x, y)
    et_sd = et_sd_list(dir_sd, start_date, end_date, x, y)
    rad = radiation_list(dir_rad, start_date, end_date, x, y)
    et_ts: list[float] = []
    for i in range(len(et_sd)):
        if i % freq == 0:
            et_ts_i = et_sd[i]
        else:
            et_ts_i = (tp[i] + sw[i] + sro[i] + src[i] + et_sd[i] + rad[i]) / 6
        et_ts.append(et_ts_i)
    return et_ts

In [ ]:
# time serie generation
et_ts = et_ts_list(
    dir_rad,
    dir_sd,
    dir_var,
    min_date,
    max_date,
    x,
    y,
    freq=3,
)

### Results

In [ ]:
plot_et_comparison(et_sd, et_ts, min_date, max_date)


In [ ]:
print_error_metrics(et_sd, et_ts, 3)